In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from pace_pic_validation.io.calcification_pic_processing import pic_from_ca

## i need volumes of filtrations to finish this


In [ ]:

BE20260729_path = '/Users/timfugl/Desktop/PhD_Ocean_Color_Chemistry/Research_Campaigns/Norbjørn /August 2026/data/BE20260729_salinity.csv'
SS20260601_path = '/Users/timfugl/Desktop/PhD_Ocean_Color_Chemistry/Obj2_CalcificationRates_of_Coccoblooms/Data/SS20260601_Incubations_no_RM.csv'
pic_path = '/Users/timfugl/Desktop/PhD_Ocean_Color_Chemistry/Obj2_CalcificationRates_of_Coccoblooms/Data/master files/Incubations_PIC.csv'

BE20260729_df = pd.read_csv(BE20260729_path)
SS20260601_df = pd.read_csv(SS20260601_path)
pic_df = pd.read_csv(pic_path)

print(pic_df.columns)

print(BE20260729_df.columns)

print(BE20260729_df.columns)


print(SS20260601_df['Sample ID'])



In [ ]:
SS20260601_df = SS20260601_df.rename(columns={'Sample ID': 'sample_id'})
SS20260601_df['date_measured_alk'] = 'NA'

parts = SS20260601_df["sample_id"].astype("string").str.extract(
    r"^(?P<date>\d{8})_(?P<station>\d+)(?P<sample>[A-D]+)_(?P<time>t[01])$"
)

parsed_ids = pd.DataFrame({
    "Cruise": "SS" + parts["date"],
    "Station": pd.to_numeric(parts["station"], errors="coerce").astype("Int64"),
    "Sample ID": parts["sample"],
    "Time ID": parts["time"],
}, index=SS20260601_df.index)


SS20260601_df = pd.concat([SS20260601_df, parsed_ids], axis=1)





SS20260601_df = SS20260601_df.drop(columns=['Unnamed: 27', 'Unnamed: 28', 'Unnamed: 29', 'Unnamed: 30',
       'Unnamed: 31', 'Unnamed: 32', 'Unnamed: 33', 'Unnamed: 34',
       'Unnamed: 35', 'Unnamed: 36', 'Unnamed: 37', 'Unnamed: 38',
       'Unnamed: 39', 'Unnamed: 40', 'Unnamed: 41', 'Unnamed: 42',
       'Unnamed: 43', 'Unnamed: 44', 'Unnamed: 45', 'Unnamed: 46',
       'Unnamed: 47', 'r2_1', 'r2_2', 'r2_3', 'r2_4','Sal1 (K*2)', 'Sal2 ( (K*2)', 'Sal3  (K*2)',
       'Sal4   (K*2)', 'Sal5  (K*2)', 'SAL_final (two best / AVG)','points_1',
       'points_2', 'points_3', 'points_4','m_1', 'm_2', 'm_3', 'm_4', 'Unnamed: 24','sample_id', 'time ID'])

BE20260729_df = BE20260729_df.drop(columns=['SAL_1 ', 'SAL_2 ',
       'SAL_3 ', 'SAL_4 ', 'SAL_5', 'SAL_mean (K)', 'm_1', 'R_1','m_2', 'R_2','m_3', 'R_3', 'm_4', 'R_4','PIC (mg/L)', 'Unnamed: 25',])


SS20260601_df = SS20260601_df.rename(columns={'Alk_1  (µmol/kg)':'Alk_1 (µmol/kg)',
                                              'Alk_2  (µmol/kg)':'Alk_2 (µmol/kg)',
                                              'Alk_3  (µmol/kg)':'Alk_3 (µmol/kg)',
                                              'Alk_4':'Alk_4 (µmol/kg)',
                                              'Time ID ': 'Time ID'
                                              })

BE20260729_df = BE20260729_df.rename(columns={'Alk_1 ':'Alk_1 (µmol/kg)',
                                              'Alk_2 ':'Alk_2 (µmol/kg)',
                                              'Alk_3':'Alk_3 (µmol/kg)',
                                              'Alk_4 ':'Alk_4 (µmol/kg)',
                                              'SAL_real': 'SAL_ppm'})




columns = ['Cruise', 'Station', 'Sample ID', 'Time ID', 'SAL_ppm',
       'Alk_1 (µmol/kg)', 'Alk_2 (µmol/kg)', 'Alk_3 (µmol/kg)',
       'Alk_4 (µmol/kg)', 'date_measured_alk','notes']



# order columns like this: Cruise, Station, Sample ID, Time ID, Alk_1 (µmol/kg), Alk_2 (µmol/kg), Alk_3 (µmol/kg), Alk_4 (µmol/kg), Salinity (K*2), PIC (mg/L)

SS20260601_df = SS20260601_df[columns]

print(BE20260729_df.columns)
print(SS20260601_df.columns)

df = pd.concat([BE20260729_df, SS20260601_df], ignore_index=True)

In [ ]:
# only keep rows where Cruise is BE20260729 or SS20260601
df = df[df['Cruise'].isin(['BE20260729', 'SS20260601'])]

print(df['Cruise'].unique())

print(df.columns)

# normalise alkalinity measurements to a common salinity of 35.0 using the formula: Alk_mean_NTA_35 = Alk_mean * (35.0 / SAL_ppm)

columns = ['Alk_1 (µmol/kg)', 'Alk_2 (µmol/kg)', 'Alk_3 (µmol/kg)',
       'Alk_4 (µmol/kg)']

for col in columns:

    colname = col + '_NTA_35'
    df[colname] = df[col] * (35.0 / df['SAL_ppm'])

# drop columns that are not needed
df = df.drop(columns=['Alk_1 (µmol/kg)', 'Alk_2 (µmol/kg)', 'Alk_3 (µmol/kg)', 'Alk_4 (µmol/kg)'])

# average Alk_values for each row
df['Alk_mean (µmol/kg)'] = df[['Alk_1 (µmol/kg)_NTA_35', 'Alk_2 (µmol/kg)_NTA_35', 'Alk_3 (µmol/kg)_NTA_35', 'Alk_4 (µmol/kg)_NTA_35']].mean(axis=1)

# std dev for alk
df['Alk_std_NTA_35 (µmol/kg)'] = df[['Alk_1 (µmol/kg)_NTA_35', 'Alk_2 (µmol/kg)_NTA_35', 'Alk_3 (µmol/kg)_NTA_35', 'Alk_4 (µmol/kg)_NTA_35']].std(axis=1)

# normalising alkalinity to common salinity of 35.0

df['Alk_mean_NTA_35 (µmol/kg)'] = df['Alk_mean (µmol/kg)'] * (35.0 / df['SAL_ppm'])

print(df.columns)


#df.to_csv('/Users/timfugl/Desktop/PhD_Ocean_Color_Chemistry/Obj2_CalcificationRates_of_Coccoblooms/Data/Incubations_alkalinity_normalised_to_salinity_35.csv', index=False)

In [ ]:
# if sample ID is AB or CD i want to create two new rows for each sample ID, one for A and one for B, and one for C and one for D, with the same values for all other columns except Sample ID which will be A or B or C or D.

df["Sample ID"] = df["Sample ID"].apply(
    lambda x: ["A", "B"] if x == "AB" else x
)

df["Sample ID"] = df["Sample ID"].apply(
    lambda x: ["C", "D"] if x == "CD" else x
)

df = df.explode("Sample ID", ignore_index=True)

df["Station"] = df["Station"].astype("Int64")

df['exp_ID']= df['Cruise'] + '_' + df['Station'].astype(str) + '_' + df['Sample ID'] 

print(df['exp_ID'].unique())


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns


# Use the final character of 'Sample ID' as the duplicate letter
df["duplicate"] = df["Sample ID"].str.strip().str[-1]


# Map duplicate bottles to depth pairs
df["depth_pair"] = df["duplicate"].map(
    {
        "A": "AB",
        "B": "AB",
        "C": "CD",
        "D": "CD",
    }
)


# Calculate mean and std per cruise-station-depth pair
summary = (
    df
    .groupby(["Cruise", "Station", "depth_pair"], as_index=False)
    .agg(
        salinity_mean=("SAL_ppm", "mean"),
        salinity_std=("SAL_ppm", "std"),
        alkalinity_mean=("Alk_mean (µmol/kg)", "mean"),
        alkalinity_std=("Alk_mean (µmol/kg)", "std"),
        n=("SAL_ppm", "size"),
    )
)


# Create a readable label for each depth pair
summary["label"] = (
    summary["Cruise"]
    + "_"
    + summary["Station"].astype(str)
    + "_"
    + summary["depth_pair"]
)


# Paste the x-axis labels you want to remove
depth_pairs_to_remove = [
    #"BE20260729_2_AB",
    #"BE20260729_5_AB",
]


summary = summary[~summary["label"].isin(depth_pairs_to_remove)].copy()


# Calculate mean standard deviations across the plotted depth pairs
mean_salinity_std = summary["salinity_std"].mean()
mean_alkalinity_std = summary["alkalinity_std"].mean()


# Create a taller two-panel figure
fig, (ax_sal, ax_alk) = plt.subplots(
    nrows=2,
    ncols=1,
    figsize=(10, 15),
    sharex=True,
)


# Salinity panel
salinity_plot = ax_sal.errorbar(
    x=summary["label"],
    y=summary["salinity_mean"],
    yerr=summary["salinity_std"],
    fmt="o",
    capsize=4,
    ecolor="black",
    elinewidth=1,
    markersize=6,
    color="tab:blue",
    label=(
        f"Mean salinity SD: {mean_salinity_std:.3f} ppm"
    ),
)


# Set salinity ticks every 0.1 ppm
sal_min = np.floor(summary["salinity_mean"].min() * 10) / 10
sal_max = np.ceil(
    (summary["salinity_mean"] + summary["salinity_std"]).max() * 10
) / 10


ax_sal.set_ylim(sal_min, sal_max)
ax_sal.set_yticks(np.arange(sal_min, sal_max + 0.1, 0.1))


ax_sal.grid(
    True,
    axis="y",
    linestyle="--",
    linewidth=0.6,
    alpha=0.5,
)


ax_sal.set_ylabel("Salinity (ppm)")
ax_sal.set_title(
    "Salinity and alkalinity consistency per depth pair",
    pad=12,
)


# Add the salinity legend
ax_sal.legend(
    loc="upper right",
    frameon=True,
    framealpha=0.9,
)


# Alkalinity panel
alkalinity_plot = ax_alk.errorbar(
    x=summary["label"],
    y=summary["alkalinity_mean"],
    yerr=summary["alkalinity_std"],
    fmt="o",
    capsize=4,
    ecolor="black",
    elinewidth=1,
    markersize=6,
    color="tab:red",
    label=(
        f"Mean alkalinity SD: {mean_alkalinity_std:.1f} µmol kg⁻¹"
    ),
)


# Set alkalinity ticks every 5 µmol kg-1
alk_min = np.floor(
    (summary["alkalinity_mean"] - summary["alkalinity_std"]).min() / 5
) * 5
alk_max = np.ceil(
    (summary["alkalinity_mean"] + summary["alkalinity_std"]).max() / 5
) * 5


ax_alk.set_ylim(alk_min, alk_max)
ax_alk.set_yticks(np.arange(alk_min, alk_max + 40, 40))


ax_alk.grid(
    True,
    axis="y",
    linestyle="--",
    linewidth=0.6,
    alpha=0.5,
)


ax_alk.set_ylabel("Alkalinity (µmol kg⁻¹)")
ax_alk.set_xlabel("Cruise, station, and depth pair")


# Add the alkalinity legend
ax_alk.legend(
    loc="upper right",
    frameon=True,
    framealpha=0.9,
)


# Make x-axis labels readable
plt.setp(ax_alk.get_xticklabels(), rotation=90, ha="center")


plt.tight_layout()


plt.savefig(
    "/Users/timfugl/Desktop/PhD_Ocean_Color_Chemistry/"
    "Obj2_CalcificationRates_of_Coccoblooms/Figures/"
    "calcification 2026/"
    "salinity_alkalinity_depth_pairs.png",
    dpi=300,
)


plt.show()

In [ ]:
keys = ['Cruise', 'Station', 'Sample ID', 'Time ID ']

merged = df.merge(pic_df, on=keys, how='inner')

merged = merged.dropna(axis=1, how='all')

print(merged.columns)


print(merged.columns)




df_t1["Alk_mean_NTA_35_diff (µmol/kg)"] = (

    df_t1["Alk_mean_NTA_35 (µmol/kg)"]
    - df_t1["exp_ID"].map(t0_alk_by_exp)
)

df_t1['delta PIC mmol/m3'] = (

    df_t1['PIC (mmol/m3)']
    - df_t1['exp_ID'].map(df_t0.set_index("exp_ID")["PIC (mmol/m3)"])

)

# Approximate density (kg/m³) = 998.2 + 0.76 × salinity
df_t1['delta PIC (µmol/kg)'] = (
    df_t1['delta PIC mmol/m3'] * 1000
    / (998.2 + 0.76 * df_t1['SAL_ppm'])
)

print(df_t1['delta PIC (µmol/kg)'])

# exclude negative values of Alk_mean_NTA_35_diff (µmol/kg) and Alk_mean_NTA_35_diff (µmol/kg) > 10

diff_col = "Alk_mean_NTA_35_diff (µmol/kg)"

print('before filtering')
print(df['Cruise'].unique())

df = df_t1.loc[(df_t1[diff_col] > 0)]
print(df['Cruise'].unique())
print(len(df))

In [ ]:
# calculate calcification rate in µmol/kg/day using the formula: calc_rate = Alk_mean_NTA_35_diff (µmol/kg) / 2 / 1 days

df["calc_rate_day"] = df[diff_col] / 2
df["calc_rate_day_std"] = df['Alk_std_NTA_35 (µmol/kg)'] / 2

print(df['Cruise'].unique())


In [ ]:
import matplotlib.pyplot as plt
import numpy as np


plot_name = "raw_data_with_pic"

# First average within each experiment, then calculate station averages
by_experiment = (
    df.groupby(["Cruise", "Station", "exp_ID"], as_index=False)
    .agg(
        calc_rate_day=("calc_rate_day", "mean"),
        pic_umol_kg=("delta PIC (µmol/kg)", "mean"),
    )
)

summary = (
    by_experiment.groupby(["Cruise", "Station"], as_index=False)
    .agg(
        calc_rate_day=("calc_rate_day", "mean"),
        calc_rate_sd=("calc_rate_day", "std"),
        pic_umol_kg=("pic_umol_kg", "mean"),
    )
)

x = np.arange(len(summary))
width = 0.4

fig, ax1 = plt.subplots(figsize=(9, 5))
ax2 = ax1.twinx()

rate_bars = ax1.bar(
    x - width / 2,
    summary["calc_rate_day"],
    width,
    color="steelblue",
    label="Calcification rate",
)

pic_bars = ax2.bar(
    x + width / 2,
    summary["pic_umol_kg"],
    width,
    color="darkorange",
    label="PIC change",
)

# Show variability between experiments at each station, when available
has_sd = summary["calc_rate_sd"].notna().to_numpy()

ax1.errorbar(
    x[has_sd] - width / 2,
    summary.loc[has_sd, "calc_rate_day"],
    yerr=summary.loc[has_sd, "calc_rate_sd"],
    fmt="none",
    capsize=4,
    color="black",
    zorder=3,
)

# Both axes start at zero
# PIC axis is 10 times smaller than the calcification-rate axis
rate_max = np.nanmax(
    (summary["calc_rate_day"] + summary["calc_rate_sd"].fillna(0)).to_numpy()
)

rate_ylim = rate_max * 1.1
pic_ylim = rate_ylim / 10

ax1.set_ylim(0, rate_ylim)
ax2.set_ylim(0, pic_ylim)

ax1.set_xticks(x)
ax1.set_xticklabels(
    summary["Cruise"].astype(str) + "\nStation " + summary["Station"].astype(str),
    rotation=90,
)

ax1.set_ylabel(
    "Calcification rate (µmol CaCO₃ kg⁻¹ day⁻¹)",
    color="steelblue",
)
ax2.set_ylabel("PIC change (µmol/kg)", color="darkorange")
ax1.set_xlabel("Station")

ax1.tick_params(axis="y", colors="steelblue")
ax2.tick_params(axis="y", colors="darkorange")

ax1.legend(
    [rate_bars, pic_bars],
    ["Calcification rate", "PIC change"],
    frameon=False,
)

fig.tight_layout()
plt.savefig(
    f"/Users/timfugl/Desktop/PhD_Ocean_Color_Chemistry/"
    f"Obj2_CalcificationRates_of_Coccoblooms/Figures/"
    f"calcification 2026/{plot_name}.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

In [ ]:
print(pic_df.columns)